# 第11周 SQL Agent 作业演示

作者：王鑫炜。使用课程原始 Chinook 数据库。本 Notebook 调用同目录实现，固定模型响应只用于离线验证。真实模型需要配置 `.env`。

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if not (ROOT / "main.py").is_file():
    raise RuntimeError("请将工作目录设为 week11 作业目录")
sys.path.insert(0, str(ROOT))
from database import Database
from llm import ReplayModel, ChatModel, load_settings
from sql_agent import SQLAgent
from main import load_cases
db = Database(ROOT / "data" / "chinook.db")
cases = {c["id"]: c for c in load_cases()}

## 数据库解析

读取字段、主外键、行数和发票日期范围，不发送原始客户或员工样本。

In [2]:
schema = db.parse()
print("业务表数量:", len(schema))
for name, info in schema.items():
    print(name, info["row_count"], info["foreign_keys"])
print("发票日期范围:", schema["invoices"]["date_range"])

业务表数量: 11
albums 347 [{'column': 'ArtistId', 'target_table': 'artists', 'target_column': 'ArtistId'}]
artists 275 []
customers 59 [{'column': 'SupportRepId', 'target_table': 'employees', 'target_column': 'EmployeeId'}]
employees 8 [{'column': 'ReportsTo', 'target_table': 'employees', 'target_column': 'EmployeeId'}]
genres 25 []
invoice_items 2240 [{'column': 'TrackId', 'target_table': 'tracks', 'target_column': 'TrackId'}, {'column': 'InvoiceId', 'target_table': 'invoices', 'target_column': 'InvoiceId'}]
invoices 412 [{'column': 'CustomerId', 'target_table': 'customers', 'target_column': 'CustomerId'}]
media_types 5 []
playlist_track 8715 [{'column': 'TrackId', 'target_table': 'tracks', 'target_column': 'TrackId'}, {'column': 'PlaylistId', 'target_table': 'playlists', 'target_column': 'PlaylistId'}]
playlists 18 []
tracks 3503 [{'column': 'MediaTypeId', 'target_table': 'media_types', 'target_column': 'MediaTypeId'}, {'column': 'GenreId', 'target_table': 'genres', 'target_column': 'Genr

## 选表 生成 校验 执行 汇总

观察按账单国家统计的销售额；直接对发票头求和，不连接明细。

In [3]:
case = cases["country_sales"]
result = SQLAgent(db, ReplayModel(case)).run(case["question"])
print("运行模式:", result["mode"])
print("选择的表:", result["selected_tables"])
print("SQL:", result["sql"])
print("真实结果:", result["result"]["rows"])
print("回答:", result["answer"])
assert result["result"]["rows"] == case["expected_rows"]

运行模式: replay
选择的表: ['invoices']
SQL: SELECT BillingCountry AS country, ROUND(SUM(Total), 2) AS revenue FROM invoices GROUP BY BillingCountry ORDER BY revenue DESC, country ASC LIMIT 5
真实结果: [['USA', 523.06], ['Canada', 303.96], ['France', 195.1], ['Brazil', 190.1], ['Germany', 156.48]]
回答: [离线回放] country, revenue=USA, 523.06；country, revenue=Canada, 303.96；country, revenue=France, 195.1；country, revenue=Brazil, 190.1；country, revenue=Germany, 156.48。统计口径：按 BillingCountry 分组，对发票头 Total 求和，使用全部历史数据；数据未注明币种。


## SQL 报错和改写

初次生成不存在的 Amount 字段。数据库真实报错，模型替身返回修正后的 Total 字段，重新校验执行。

In [4]:
case = cases["rewrite_column"]
fixed = SQLAgent(db, ReplayModel(case)).run(case["question"])
for event in fixed["trace"]:
    print(json.dumps(event, ensure_ascii=False))
assert fixed["rewrites"] == 1
assert fixed["result"]["rows"] == [[2328.6]]
print("最终结果:", fixed["result"]["rows"])

{"stage": "database_parse", "status": "ok", "table_count": 11}
{"stage": "table_select", "status": "ok", "requested_tables": ["invoices"], "tables": ["invoices"], "reason": "预设演示表"}
{"stage": "sql_generate", "status": "ok", "sql": "SELECT ROUND(SUM(Amount), 2) AS revenue FROM invoices", "explanation": "发票头每行一张发票，对 Total 求和；初始 SQL 故意使用不存在的 Amount 演示自动修正。"}
{"stage": "sql_validate", "status": "error", "error": "no such column: Amount", "sql": "SELECT ROUND(SUM(Amount), 2) AS revenue FROM invoices"}
{"stage": "sql_rewrite", "status": "ok", "sql": "SELECT ROUND(SUM(Total), 2) AS revenue FROM invoices", "explanation": "根据实际数据库报错修正字段或表达式"}
{"stage": "sql_validate", "status": "ok", "plan": [[3, 0, 102, "SCAN invoices"]]}
{"stage": "sql_execute", "status": "ok", "returned_rows": 1, "truncated": false}
{"stage": "result_summary", "status": "ok"}
最终结果: [[2328.6]]


## 多表 空结果和歧义澄清

课程库发票数据覆盖 2009—2013 年；空记录不能代表各国销售额均为零。

In [5]:
for case_id in ("genre_sales", "bridge_join", "empty_result", "clarification"):
    case = cases[case_id]
    out = SQLAgent(db, ReplayModel(case)).run(case["question"])
    print(case_id, out["status"], out["selected_tables"], out["answer"])

genre_sales ok ['genres', 'invoice_items', 'tracks'] [离线回放] genre, units, revenue=Rock, 835, 826.65；genre, units, revenue=Latin, 386, 382.14；genre, units, revenue=Metal, 264, 261.36。统计口径：按明细成交单价乘数量计算销售额，按 Quantity 求销量；tracks 仅用于关联流派。
bridge_join ok ['albums', 'artists', 'invoice_items', 'tracks'] [离线回放] artist, revenue=Iron Maiden, 138.6；artist, revenue=U2, 105.93；artist, revenue=Metallica, 90.09。统计口径：表选择故意只返回两端，程序沿外键补全 albums 和 tracks；在明细粒度统计成交额。
empty_result ok ['invoices'] [离线回放] 查询没有匹配记录。
clarification needs_clarification [] 请明确时间范围，以及要看销售额还是售出数量。当前发票数据覆盖 2009—2013 年，是否分析其中某个时间段？


## 真实模型入口

配置 `.env` 后把 RUN_LIVE 改为 True。调用会把问题、结构和结果发送到配置的模型服务；保存的 Notebook 输出不代表真实模型验收。

In [6]:
RUN_LIVE = False
if RUN_LIVE:
    model = ChatModel(load_settings(ROOT / ".env"))
    live = SQLAgent(db, model).run("2013 年每月的发票销售额分别是多少？")
    print(json.dumps(live, ensure_ascii=False, indent=2))
else:
    print("真实模型调用未启用；请先配置 .env。")

真实模型调用未启用；请先配置 .env。


## 理解检查

1. 为什么有专辑艺术家数是 204，而 artists 表有 275 行？
2. 为什么按流派统计销售额不能连接明细后对发票头 Total 求和？
3. EXPLAIN 通过能否证明业务口径正确？
4. 把最大行数设为 2，解释 truncated 与总行数的区别。

先凭记忆解释，再查看代码与原始结果。